## **Run Code Below to Mount Drive before starting anything in notebook**

In [ ]:
# Ensure Google Drive is mounted if you are running this cell in a new session
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!ls "path/to/project_folder"

### 1. Identifying Station within a set one mile radius around closed 14th St and 23rd St FM stations

In [ ]:
import pandas as pd

file_path_3 = "path/to/project_folder/stops.txt"
stops_df = pd.read_csv(file_path_3)
print(stops_df.head())
print(stops_df.info())



In [ ]:
# filter on location_type "1"

# Filter the dataframe based on `location_type`
filtered_stops_df = stops_df[stops_df['location_type'] == 1].copy()

# Display the first few rows and info of the filtered dataframe
print("\nFiltered stops_df (location_type = 1):")
print(filtered_stops_df.head())
print(filtered_stops_df.info())

In [ ]:
import pandas as pd
import numpy as np

# Load the MTA_Subway_Stations_and_Complexes file
mta_stations_complexes_path = "path/to/project_folder/MTA_Subway_Stations_and_Complexes.csv"
mta_stations_complexes_df = pd.read_csv(mta_stations_complexes_path)
mta_stations_complexes_df = mta_stations_complexes_df[['Complex ID', 'Station IDs', 'GTFS Stop IDs']].copy()

# Break apart 'GTFS Stop IDs' by semicolon and remove spaces
gtfs_ids_split = mta_stations_complexes_df['GTFS Stop IDs'].str.split(';', expand=True)

# Remove leading/trailing spaces from the split IDs
gtfs_ids_split = gtfs_ids_split.apply(lambda x: x.str.strip())

# Create new columns for each split ID
# The number of new columns will be the maximum number of split IDs found in any row
num_cols = gtfs_ids_split.shape[1]
new_col_names = [f'GTFS_Stop_ID_{i+1}' for i in range(num_cols)]
gtfs_ids_split.columns = new_col_names

# Concatenate the new columns back to the original DataFrame
mta_stations_complexes_df = pd.concat([mta_stations_complexes_df, gtfs_ids_split], axis=1)

# Ensure the GTFS Stop ID columns in mta_stations_complexes_df are of object/string type
gtfs_id_columns = [col for col in mta_stations_complexes_df.columns if 'GTFS_Stop_ID' in col]
for col in gtfs_id_columns:
    mta_stations_complexes_df[col] = mta_stations_complexes_df[col].astype(str).str.strip()

# Ensure the stop_id column in stops_within_1_mile_radius is of object/string type
filtered_stops_df['stop_id'] = filtered_stops_df['stop_id'].astype(str).str.strip()

# Create a dictionary mapping GTFS Stop IDs to Complex IDs from the complexes dataframe
# We need to build this dictionary from all the GTFS Stop ID columns
gtfs_to_complex_map = {}
for index, row in mta_stations_complexes_df.iterrows():
    complex_id = row['Complex ID']
    # Iterate through all GTFS Stop ID columns for this row
    for col in gtfs_id_columns:
        stop_id = row[col]
        # Add to map if the stop_id is not None, not an empty string, and not already mapped
        if pd.notna(stop_id) and stop_id != '' and stop_id != 'nan':
             # Handle potential multiple Complex IDs for the same GTFS Stop ID
             # Here we just take the first one encountered.
             if stop_id not in gtfs_to_complex_map:
                gtfs_to_complex_map[stop_id] = complex_id


# Function to lookup Complex ID based on stop_id
def lookup_complex_id(stop_id, gtfs_to_complex_map):
    """
    Looks up the Complex ID for a given stop_id using the pre-built map.
    """
    return gtfs_to_complex_map.get(stop_id, np.nan) # Return np.nan if stop_id not found

# Apply the lookup function to the 'stop_id' column of stops_within_1_mile_radius
filtered_stops_df['Complex ID'] = filtered_stops_df['stop_id'].apply(
    lambda x: lookup_complex_id(x, gtfs_to_complex_map)
)

# Display the resulting DataFrame with the new 'Complex ID' column
print("\nStops within 1 mile radius with Complex ID:")
print(filtered_stops_df.head())
print(filtered_stops_df.info())


In [ ]:
# filter for the missing complex ID

# Filter for rows where 'Complex ID' is missing (NaN)
missing_complex_id_stops = filtered_stops_df[filtered_stops_df['Complex ID'].isna()]

# Get the unique station_complex names for the stops with missing Complex IDs
stations_with_missing_complex_id = missing_complex_id_stops[['stop_id', 'stop_name']].drop_duplicates()

print("\nStations with missing Complex IDs:")
stations_with_missing_complex_id


In [ ]:
# drop n/a based on complex ID and drop the parent staiton column

# Drop rows where 'Complex ID' is NaN
filtered_stops_df_cleaned = filtered_stops_df.dropna(subset=['Complex ID']).copy()

# Drop the original 'parent_station' column as it's not needed after filtering by location_type
if 'parent_station' in filtered_stops_df_cleaned.columns:
    filtered_stops_df_cleaned = filtered_stops_df_cleaned.drop(columns=['parent_station'])

print("\nDataFrame after dropping rows with missing Complex ID and 'parent_station':")
print(filtered_stops_df_cleaned.head())
print(filtered_stops_df_cleaned.info())


In [ ]:
!pip install haversine --quiet

In [ ]:
# calculate the distance in miles using stop_lat and stop_long in the filtered_stops_df_cleaned for station distance between two different stations on the dataframe. One stations has a unique stop ID of D19 and the other has a complex ID 228. Create 2 columns that calculate the distance between different stations from these 2 stations

from haversine import haversine, Unit

# Define the coordinates for the two reference stations
# Assuming stop_lat and stop_long are the columns for latitude and longitude respectively
station_d19_coords = filtered_stops_df_cleaned[filtered_stops_df_cleaned['stop_id'] == 'D19'][['stop_lat', 'stop_lon']].iloc[0]
station_228_coords = filtered_stops_df_cleaned[filtered_stops_df_cleaned['Complex ID'] == 228][['stop_lat', 'stop_lon']].iloc[0]

# Convert pandas series to tuple (lat, lon)
coords_d19 = (station_d19_coords['stop_lat'], station_d19_coords['stop_lon'])
coords_228 = (station_228_coords['stop_lat'], station_228_coords['stop_lon'])

# Function to calculate distance from a given point to a reference point
def calculate_distance(row_coords, ref_coords):
    """Calculates haversine distance in miles between two coordinate pairs."""
    # Ensure both are tuples (lat, lon)
    row_tuple = (row_coords['stop_lat'], row_coords['stop_lon'])
    ref_tuple = (ref_coords[0], ref_coords[1])
    return haversine(row_tuple, ref_tuple, unit=Unit.MILES)

# Apply the function to create new columns for distance to each reference station
# Use .copy() to avoid SettingWithCopyWarning
filtered_stops_df_cleaned['Distance_to_D19_miles'] = filtered_stops_df_cleaned.apply(
    lambda row: calculate_distance(row[['stop_lat', 'stop_lon']], coords_d19),
    axis=1
)

filtered_stops_df_cleaned['Distance_to_228_miles'] = filtered_stops_df_cleaned.apply(
    lambda row: calculate_distance(row[['stop_lat', 'stop_lon']], coords_228),
    axis=1
)

# Display the updated DataFrame with the new distance columns
print("\nDataFrame with distance calculations:")
print(filtered_stops_df_cleaned[['stop_id', 'Complex ID', 'stop_lat', 'stop_lon', 'Distance_to_D19_miles', 'Distance_to_228_miles']].head())
print(filtered_stops_df_cleaned.info())

In [ ]:
# filter on complex 601 and 228

# Define the complex IDs you want to filter on
complex_id_601 = 601
complex_id_228 = 228

# Filter the DataFrame to include only rows where 'Complex ID' is either 601 or 228
filtered_complexes_df = filtered_stops_df_cleaned[
    (filtered_stops_df_cleaned['Complex ID'] == complex_id_601) |
    (filtered_stops_df_cleaned['Complex ID'] == complex_id_228)
].copy() # Use .copy() to avoid SettingWithCopyWarning

# Display the first few rows and info of the filtered dataframe
print(f"\nFiltered DataFrame showing only Complex IDs {complex_id_601} and {complex_id_228}:")
print(filtered_complexes_df.head())
print(filtered_complexes_df.info())

In [ ]:
# create two filtered lists: one that filters the column Distance_to_D19_miles for anything in a half mile radius and another that filters Distance_to_228_miles for anything in a half mile radius. respectively drop the others column

# Define the radius (1 miles)
radius_miles = 1

# Filter for stops within a half-mile radius of D19
stops_near_d19 = filtered_stops_df_cleaned[
    filtered_stops_df_cleaned['Distance_to_D19_miles'] <= radius_miles
].copy()

# Filter for stops within a half-mile radius of Complex 228
stops_near_228 = filtered_stops_df_cleaned[
    filtered_stops_df_cleaned['Distance_to_228_miles'] <= radius_miles
].copy()

# Drop the other distance column from each filtered dataframe
stops_near_d19 = stops_near_d19.drop(columns=['Distance_to_228_miles'])
stops_near_228 = stops_near_228.drop(columns=['Distance_to_D19_miles'])

print(f"\nStations within {radius_miles} miles of D19:")
print(stops_near_d19.head())
print(stops_near_d19.info())

print(f"\nStations within {radius_miles} miles of Complex 228:")
print(stops_near_228.head())
print(stops_near_228.info())

In [ ]:
#export the data in both the stops_near_d19 and stops_near_228 to "path/to/project_folder"

output_dir = "path/to/project_folder"

# Export stops_near_d19 to a CSV file
stops_near_d19.to_csv(output_dir + "stops_near_d19.csv", index=False)
print(f"\nExported stops_near_d19 to {output_dir}stops_near_d19.csv")

# Export stops_near_228 to a CSV file
stops_near_228.to_csv(output_dir + "stops_near_228.csv", index=False)
print(f"\nExported stops_near_228 to {output_dir}stops_near_228.csv")

In [ ]:
# using the combined unique IDs create a df labeled stops_within_1_mile_radius that contains the stop_id,  stop_name ,  stop_lat , stop_lon columns found both in stops_near_d19 and stops_near_228 but also include the radius data of both datasets as well found under the Distance_to_D19_miles  and Distance_to_228_miles columns from the same dataset

# Get the stop_ids from stops_near_d19 and stops_near_228
stop_ids_d19 = stops_near_d19['stop_id'].tolist()
stop_ids_228 = stops_near_228['stop_id'].tolist()

# Combine the lists
combined_stop_ids = stop_ids_d19 + stop_ids_228

# Remove duplicates by converting to a set and then back to a list
unique_combined_stop_ids = list(set(combined_stop_ids))
len(unique_combined_stop_ids)

# Filter the original filtered_stops_df_cleaned to get rows for these unique stop IDs
stops_within_1_mile_radius = filtered_stops_df_cleaned[
    filtered_stops_df_cleaned['stop_id'].isin(unique_combined_stop_ids)
][['stop_id','Complex ID', 'stop_name', 'stop_lat', 'stop_lon', 'Distance_to_D19_miles', 'Distance_to_228_miles']].copy()

print("\nDataFrame 'stops_within_1_mile_radius':")
print(stops_within_1_mile_radius.head())
print(stops_within_1_mile_radius.info())




### 2. EDA Overview on MTA data

Focus placed on 2024 and random data points in a Monday through Friday work week evening rush times of 4 PM – 7 PM

#### 2.1 Random Selection of MTA 2024 Data Points based on Station Complexs identified within a mile radius of 14th and 23rd St station : **<font color="red"> Code below created randomized MTA 2024 Dataset for 1 random week within each individual month of the year. Do not rerun code unless needed so to not overwrite already created CSV dataset. </font>**

In [ ]:
'''
!pip install sodapy pandas numpy tqdm --quiet

import pandas as pd
import numpy as np
from sodapy import Socrata
import time
from tqdm import tqdm
from datetime import datetime

# ====================== CONFIGURATION ======================
API_TOKEN = "YOUR_API_TOKEN_HERE"
DATASET_ID = "wujg-7c2s"

# Replace with your actual list of float IDs from unique_combined_complex_ids
UNIQUE_COMBINED_COMPLEX_IDS = [unique_combined_complex_ids]  # Example float IDs

STATION_COMPLEX_IDS = [
    str(int(float(item)))
    for sublist in UNIQUE_COMBINED_COMPLEX_IDS
    for item in (sublist if isinstance(sublist, list) else [sublist])
]
# ============================================================

# Initialize API client
client = Socrata(
    "data.ny.gov",
    API_TOKEN,
    timeout=30
)

def get_monthly_work_weeks():
    """Generate random work weeks with validation"""
    sampled_days = []

    for month in range(1, 13):
        month_start = f"2024-{month:02d}-01"
        end_date = pd.Timestamp(month_start) + pd.offsets.MonthEnd(1)
        dates = pd.date_range(start=month_start, end=end_date, freq='D')

        valid_weeks = []
        for date in dates:
            if date.weekday() == 0:  # Monday
                week = [date + pd.DateOffset(days=i) for i in range(5)]
                if all(day.month == month for day in week):
                    valid_weeks.append(week)

        if valid_weeks:
            selected_week = valid_weeks[np.random.randint(len(valid_weeks))]
            sampled_days.extend(selected_week)

    return [day.to_pydatetime() for day in sampled_days]

def fetch_time_window(date_obj, start_time, end_time):
    """Robust API fetching with error handling"""
    date_str = date_obj.strftime("%Y-%m-%d")
    station_ids_str = ', '.join([f"'{id}'" for id in STATION_COMPLEX_IDS])
    where_clause = (
        f"transit_timestamp BETWEEN '{date_str}T{start_time}' AND '{date_str}T{end_time}' "
        f"AND station_complex_id IN ({station_ids_str})"
    )

    results = []
    offset = 0
    limit = 50000

    try:
        with tqdm(desc=f"Fetching {date_str} {start_time}-{end_time}") as pbar:
            while True:
                batch = client.get(
                    DATASET_ID,
                    where=where_clause,
                    limit=limit,
                    offset=offset
                )

                if isinstance(batch, dict):
                    print(f"API Error: {batch.get('message', 'Unknown error')}")
                    break

                if not batch:
                    break

                results.extend(batch)
                offset += limit
                pbar.update(len(batch))
                time.sleep(1)

    except Exception as e:
        print(f"\nCritical Error: {str(e)}")

    return results

def process_geodata(record):
    """Standardize georeference data with error handling"""
    try:
        geo = record.pop('georeference', None) or record.pop('Georeference', None)

        if isinstance(geo, dict):
            record.update({
                'geo_type': geo.get('type'),
                'longitude': geo.get('coordinates', [None, None])[0],
                'latitude': geo.get('coordinates', [None, None])[1]
            })
        elif isinstance(geo, str) and geo.startswith("POINT"):
            coords = geo[6:-1].strip().split()
            record.update({
                'longitude': coords[0],
                'latitude': coords[1]
            })

    except Exception as e:
        print(f"Error processing geodata: {e}")

    return record

def sanitize_record(record):
    """Remove any remaining nested structures"""
    return {
        key: str(value) if isinstance(value, (dict, list)) else value
        for key, value in record.items()
    }

# Main execution
if __name__ == "__main__":
    # Fetch and process data
    sampled_days = get_monthly_work_weeks()

    time_windows = {
        "evening_rush": ("16:00:00", "19:00:00")
    }

    all_data = []
    for day in sampled_days:
        for start, end in time_windows.values():
            all_data.extend(fetch_time_window(day, start, end))

    # Process records with sanitization
    processed_data = [
        sanitize_record(process_geodata(record))
        for record in all_data
    ]

    # Create DataFrame
    df = pd.DataFrame(processed_data).drop_duplicates()

    # Updated column mapping with additional fields
    column_map = {
        'station_complex_id': ['station_complex_id'],
        'ridership': ['ridership', 'riders', 'entries', 'exits'],
        'transfers': ['transfers', 'transfer_count'],
        'latitude': ['latitude', 'lat'],
        'longitude': ['longitude', 'lon', 'lng'],
        'transit_timestamp': ['transit_timestamp'],
        'transit_mode': ['transit_mode'],
        'station_complex': ['station_complex'],
        'borough': ['borough'],
        'payment_method': ['payment_method'],
        'fare_class_category': ['fare_class_category']
    }

    standardized_df = pd.DataFrame()
    for new_col, old_cols in column_map.items():
        for col in old_cols:
            if col in df.columns:
                standardized_df[new_col] = df[col]
                break
        else:
            standardized_df[new_col] = np.nan

    # Numeric conversion
    numeric_cols = ['station_complex_id', 'ridership', 'transfers', 'latitude', 'longitude']
    standardized_df[numeric_cols] = standardized_df[numeric_cols].apply(
        pd.to_numeric, errors='coerce'
    )

    # Convert timestamp
    if 'transit_timestamp' in standardized_df:
        standardized_df['transit_timestamp'] = pd.to_datetime(
            standardized_df['transit_timestamp'], errors='coerce'
        )

    # Final validation
    if standardized_df.empty:
        print("\n⚠️ Warning: No data retrieved. Check:")
        print("- Station IDs exist")
        print("- Date filters are within dataset range")
        print("- API token is valid")
    else:
        standardized_df.to_parquet("mta_station_data.parquet")
        print(f"\n✅ Success! Processed {len(standardized_df):,} records")
'''

#### 2.2 MTA data exported into CSV and evaluated

In [ ]:
'''
# export the standardized_df to a csv file labeled MTA 2024 Data 1 Mile Radius file_path = "path/to/project_folder"
file_path = "path/to/project_folder"
output_file_path = file_path + "/MTA 2024 Data 1 Mile Radius.csv"
standardized_df.to_csv(output_file_path, index=False)
print(f"\nExported standardized_df to {output_file_path}")
'''

In [ ]:
#  import the data as a df labled MTA 2024 Data Radius
file_path = "path/to/project_folder"
output_file_path = file_path + "/MTA 2024 Data 1 Mile Radius.csv"

import pandas as pd
MTA_2024_Data_Radius = pd.read_csv(output_file_path)
print(MTA_2024_Data_Radius.head())
print(MTA_2024_Data_Radius.info())
MTA_2024_Data_Radius.shape

#### 2.3 EDA for 1 Mile Radius Stations (14-FM and 23-FM St Station)

In [ ]:
# create a column that is labeled as "Month" and pull the month from the transit_timestamp column

import pandas as pd
MTA_2024_Data_Radius['Month'] = pd.to_datetime(MTA_2024_Data_Radius['transit_timestamp']).dt.month

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Ensure 'transit_timestamp' is in datetime format and extract the date
MTA_2024_Data_Radius['transit_timestamp'] = pd.to_datetime(MTA_2024_Data_Radius['transit_timestamp'])
MTA_2024_Data_Radius['Date'] = MTA_2024_Data_Radius['transit_timestamp'].dt.date

# Calculate total ridership per day from the new dataset
daily_ridership_radius = MTA_2024_Data_Radius.groupby('Date')['ridership'].sum().reset_index()
daily_ridership_radius.columns = ['Date', 'Total_Daily_Ridership']

# Convert 'Date' back to datetime for easier week calculation
daily_ridership_radius['Date'] = pd.to_datetime(daily_ridership_radius['Date'])

# Determine week number for each date (assuming 5 workdays per week in order)
daily_ridership_radius = daily_ridership_radius.sort_values(by='Date').reset_index(drop=True)
daily_ridership_radius['Week_Number'] = (daily_ridership_radius.index // 5) + 1 # Assumes 5 days per week in order

# Create the table format for weekly ridership by day
week_day_ridership_radius = daily_ridership_radius.pivot_table(
    index='Week_Number',
    columns=daily_ridership_radius.groupby('Week_Number').cumcount().add(1),
    values='Total_Daily_Ridership',
    aggfunc='sum'
)

# Rename columns to represent days (Day 1, Day 2, ..., Day 5)
week_day_ridership_radius.columns = [f'Day {i}' for i in week_day_ridership_radius.columns]

# Melt the dataframe to long format for plotting
week_day_ridership_melted_radius = week_day_ridership_radius.reset_index().melt(
    id_vars='Week_Number', var_name='Day_of_Week', value_name='Total_Ridership'
)

# Convert 'Day_of_Week' to a categorical type with the correct order for plotting
week_day_ridership_melted_radius['Day_of_Week'] = pd.Categorical(
    week_day_ridership_melted_radius['Day_of_Week'],
    categories=[f'Day {i}' for i in range(1, 6)],
    ordered=True
)

# Sort by Week_Number and Day_of_Week to ensure correct line drawing
week_day_ridership_melted_radius = week_day_ridership_melted_radius.sort_values(by=['Week_Number', 'Day_of_Week'])

# Create the line graph
plt.figure(figsize=(12, 7))

# Plot each week as a separate line
for week_number in week_day_ridership_melted_radius['Week_Number'].unique():
    week_data = week_day_ridership_melted_radius[week_day_ridership_melted_radius['Week_Number'] == week_number]
    plt.plot(
        week_data['Day_of_Week'],
        week_data['Total_Ridership'],
        marker='o', # Add markers for data points
        linestyle='-', # Draw lines between points
        label=f'Month {week_number}'
    )

# Customize plot
plt.title('Work Week Evening Ridership Trend by Day of Week in Month (1-Mile Radius Stations)', fontsize=14)
plt.xlabel('Day of Work Week (Day 1 = Monday, Day 5 = Friday)', fontsize=12)
plt.ylabel('Total Ridership', fontsize=12)
plt.xticks(rotation=0) # Keep day labels horizontal
plt.grid(axis='y', alpha=0.5, linestyle='--') # Add horizontal grid lines

# Add a legend to identify the lines (weeks)
plt.legend(title='Month Number', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming MTA_2024_Data_Radius is already loaded into a pandas DataFrame

# Ensure 'transit_timestamp' is in datetime format and extract hour and date
MTA_2024_Data_Radius['transit_timestamp'] = pd.to_datetime(MTA_2024_Data_Radius['transit_timestamp'])
MTA_2024_Data_Radius['hour'] = MTA_2024_Data_Radius['transit_timestamp'].dt.hour
MTA_2024_Data_Radius['date'] = MTA_2024_Data_Radius['transit_timestamp'].dt.date

# Determine the unique dates in the dataset
unique_dates = sorted(MTA_2024_Data_Radius['date'].unique())

# Define the start and end hours for the histograms
start_hour = 16  # 4 PM
end_hour = 19   # 7 PM

# Determine the number of weeks based on the unique dates (assuming 5 workdays per week)
num_weeks = len(unique_dates) // 5

# Create a figure and axes for the subplots
fig, axes = plt.subplots(num_weeks, 1, figsize=(15, 5 * num_weeks), squeeze=False)
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

# Loop through each week
for week_index in range(num_weeks):
    # Determine the dates for the current week
    week_dates = unique_dates[week_index * 5 : (week_index + 1) * 5]

    # Filter data for the current week and specified hours
    weekly_data = MTA_2024_Data_Radius[
        (MTA_2024_Data_Radius['date'].isin(week_dates)) &
        (MTA_2024_Data_Radius['hour'].between(start_hour, end_hour))
    ]

    # Aggregate hourly data for the entire week
    # Group by hour across all days in the week
    hourly_counts = weekly_data.groupby('hour')['ridership'].sum().reindex(
        range(start_hour, end_hour + 1), fill_value=0
    )

    # Get the current axis
    ax = axes[week_index]

    # Plot the histogram (bar chart of hourly totals)
    bars = ax.bar(
        [f"{h%12 if h%12 != 0 else 12}:00 {'AM' if h < 12 else 'PM'}" for h in hourly_counts.index],
        hourly_counts.values,
        width=0.6,
        color=plt.cm.viridis(week_index / num_weeks), # Use a color gradient for different weeks
        edgecolor='black'
    )

    # Customize labels and title
    ax.set_title(f'Hourly Ridership for 1-Mile Radius Stations - for Work Week in Month {week_index + 1}', fontsize=12, pad=12)
    ax.set_xlabel('Time of Day', fontsize=10, labelpad=8)
    ax.set_ylabel('Total Riders for Week', fontsize=10, labelpad=8)

    # Rotate x-labels for readability
    plt.sca(ax) # Set current axis for xticks
    plt.xticks(rotation=45, ha='right', fontsize=9)

    # Remove top/right borders
    ax.spines[['top', 'right']].set_visible(False)

    # Add light gridlines
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)

    # Add bar value labels
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height,
                f'{int(height)}',
                ha='center',
                va='bottom',
                fontsize=8,
                rotation=45, # Rotate labels for horizontal bars if needed
                color='black'
            )
    ax.margins(y=0.1) # Add some margin above the bars


# Adjust layout and display the plots
plt.tight_layout(pad=3.0)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Ensure 'transit_timestamp' is in datetime format and extract hour
MTA_2024_Data_Radius['transit_timestamp'] = pd.to_datetime(MTA_2024_Data_Radius['transit_timestamp'])
MTA_2024_Data_Radius['hour'] = MTA_2024_Data_Radius['transit_timestamp'].dt.hour

# Define the start and end hours for aggregation
start_hour = 16  # 4 PM
end_hour = 19   # 7 PM


# Calculate hourly ridership aggregated across all dates
hourly_ridership_agg = MTA_2024_Data_Radius.groupby('hour')['ridership'].sum().reindex(
    range(start_hour, end_hour + 1), fill_value=0
)

print("\nAggregated Hourly Ridership (4 PM - 7 PM):")
print(hourly_ridership_agg)

# Calculate the percentage change between consecutive hours
# Shift the series to get the previous hour's ridership
previous_hour_ridership = hourly_ridership_agg.shift(1)

# Calculate the percentage difference
# Handle division by zero or when previous_hour_ridership is 0 or NaN
percentage_difference = (
    (hourly_ridership_agg - previous_hour_ridership) / previous_hour_ridership * 100
)

# The first hour (4 PM) doesn't have a previous hour in this window, so its difference is NaN
percentage_difference.iloc[0] = np.nan

print("\nPercentage Difference in Ridership Between Consecutive Hours (4 PM - 7 PM):")
percentage_difference

In [ ]:
#  list out the max daily ridership for each station

import pandas as pd
# Ensure 'transit_timestamp' is in datetime format and extract the date
MTA_2024_Data_Radius['transit_timestamp'] = pd.to_datetime(MTA_2024_Data_Radius['transit_timestamp'])
MTA_2024_Data_Radius['Date'] = MTA_2024_Data_Radius['transit_timestamp'].dt.date

# Group by station_complex and Date, then sum ridership to get daily ridership per station
daily_ridership_per_station = MTA_2024_Data_Radius.groupby(['station_complex', 'Date'])['ridership'].sum().reset_index()

# Find the maximum daily ridership for each station_complex
max_daily_ridership_per_station = daily_ridership_per_station.groupby('station_complex')['ridership'].max().reset_index()

# Rename the columns for clarity
max_daily_ridership_per_station.columns = ['Station Complex', 'Max Daily Ridership']

# Print the resulting table
print("\nMaximum Daily Ridership per Station Complex (within 1-mile radius):")
print(max_daily_ridership_per_station.sort_values(by='Max Daily Ridership', ascending=False))

# Export the max daily ridership data
# Corrected the output directory path
output_dir = "path/to/project_folder/"
max_daily_ridership_per_station.to_csv(output_dir + "max_station_evening_daily_ridership.csv", index=False)

In [ ]:

# Calculate total ridership by station and include Complex ID
ridership_by_station_complex = MTA_2024_Data_Radius.groupby(['station_complex_id', 'station_complex'])['ridership'].sum().reset_index()

# Rename columns for clarity
ridership_by_station_complex.columns = ['Complex ID', 'Station Complex Name', 'Total Ridership by Station Complex']

# Sort by total ridership for better readability
ridership_by_station_complex = ridership_by_station_complex.sort_values(by='Total Ridership by Station Complex', ascending=False)


# Add the new column 'Total Daily Average Ridership'
ridership_by_station_complex['Est. Avrg WrkWk Evening Ridership'] = round(ridership_by_station_complex['Total Ridership by Station Complex'] / 12, ndigits=0)


# Add the new column 'Total Daily Average Ridership'
ridership_by_station_complex['Est. Avrg Dly WrkWk Evening Ridership'] = round(ridership_by_station_complex['Total Ridership by Station Complex'] / 60, ndigits=0)


# Display the updated table
print("\nTotal Ridership and Daily Average by Station Complex (within 1-mile radius):")
ridership_by_station_complex



In [ ]:
#  for stops_within_1_mile_radius dataframe add 3 new column Station Complex Name	Total Ridership by Station Complex	Total Daily Average Ridership by Station Complex by pulling the data from the ridership_by_station_complex

import pandas as pd
# Ensure 'Complex ID' columns are of the same type (e.g., integer or string)
# Assuming 'Complex ID' in stops_within_1_mile_radius is already numeric (from previous steps)
# Convert 'Complex ID' in ridership_by_station_complex to numeric if it's not already
ridership_by_station_complex['Complex ID'] = pd.to_numeric(ridership_by_station_complex['Complex ID'], errors='coerce')

# Merge stops_within_1_mile_radius with ridership_by_station_complex
stops_within_1_mile_radius = pd.merge(
    stops_within_1_mile_radius,
    ridership_by_station_complex[['Complex ID', 'Station Complex Name', "Est. Avrg WrkWk Evening Ridership","Est. Avrg Dly WrkWk Evening Ridership" ]],
    on='Complex ID',
    how='left' # Use a left merge to keep all stops_within_1_mile_radius rows
)

# Display the resulting DataFrame with the new columns
print("\n'stops_within_1_mile_radius' with merged ridership data:")
print(stops_within_1_mile_radius.head())
print(stops_within_1_mile_radius.info())

In [ ]:
#for stops_within_1_mile_radius updated column name Distance_to_D19_miles to be Distance to 14th St Stop in mi and update Distance_to_228_miles  to Distance to 23th St Stop in mi

stops_within_1_mile_radius = stops_within_1_mile_radius.rename(
    columns={
        'Distance_to_D19_miles': 'Distance to 14th St Stop in mi',
        'Distance_to_228_miles': 'Distance to 23rd St Stop in mi'
    }
)

print("\n'stops_within_1_mile_radius' with updated column names:")
print(stops_within_1_mile_radius.head())
print(stops_within_1_mile_radius.info())

In [ ]:
# For columns   Distance to 14th St Stop in mi and Distance to 23rd St Stop in mi review each column and identify any distance over 1 mile if over 1 mile update the cell to be "Outside of 1 Mile Radius"

import pandas as pd
# List of columns to check and update
distance_cols = ['Distance to 14th St Stop in mi', 'Distance to 23rd St Stop in mi']

# Define the radius threshold in miles
radius_threshold = 1

# Iterate through the specified columns
for col in distance_cols:
    # Ensure the column exists in the DataFrame
    if col in stops_within_1_mile_radius.columns:
        # Apply the condition: if distance > radius_threshold, update the cell
        stops_within_1_mile_radius[col] = stops_within_1_mile_radius[col].apply(
            lambda x: "Outside of 1 Mile Radius" if pd.notna(x) and isinstance(x, (int, float)) and x > radius_threshold else x
        )
    else:
        print(f"Warning: Column '{col}' not found in the DataFrame.")

# Display the updated DataFrame to verify the changes
print("\nDataFrame after checking and updating distance columns:")
print(stops_within_1_mile_radius[['stop_id', 'stop_name'] + distance_cols].head())
print(stops_within_1_mile_radius[['stop_id', 'stop_name'] + distance_cols].info())

In [ ]:
#  export to stops_within_1_mile_radius

output_dir = "path/to/project_folder"

# Export stops_within_1_mile_radius to a CSV file
stops_within_1_mile_radius.to_csv(output_dir + "stops_within_1_mile_radius.csv", index=False)
print(f"\nExported stops_within_1_mile_radius to {output_dir}stops_within_1_mile_radius.csv")

#### 2.4 Station mapping in NYC

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import re
import math  # Added for trigonometric functions
from matplotlib.patches import Ellipse  # Added for drawing circles

# --- Data Loading & Cleaning ---
file_path = "path/to/project_folder"

# Load station data with status
stations_df = pd.read_excel(
    f"{file_path}/Mapped Stops Within 1 Mile Radius.xlsx",
    sheet_name="stops_within_1_mile_radius (2)",
    usecols=["Name on Map", "stop_lat", "stop_lon",
            "Est. Avrg Dly WrkWk Evening Ridership", "On Subway Map"]
)

# Load connection data
connections_df = pd.read_excel(
    f"{file_path}/Subway and Walking Distance Between Specific Stations.xlsx",
    sheet_name="Sheet1"
)

# Clean names and mark closed stops
closed_stops = ["Closed Stop: 14-FM", "Closed Stop: 23-FM"]

stations_df["status"] = stations_df["Name on Map"].apply(
    lambda x: "closed" if x in closed_stops else "open"
)

stations_df["Name on Map"] = stations_df["Name on Map"].str.replace(" - ", "-", regex=False)
connections_df["From Station"] = connections_df["From Station"].str.replace(" - ", "-", regex=False)
connections_df["To Station"] = connections_df["To Station"].str.replace(" - ", "-", regex=False)

# --- Graph Construction ---
G = nx.MultiDiGraph()

# Add nodes with full metadata
for _, row in stations_df.iterrows():
    G.add_node(
        row["Name on Map"],
        lat=row["stop_lat"],
        lon=row["stop_lon"],
        ridership=row["Est. Avrg Dly WrkWk Evening Ridership"],
        status=row["status"]
    )

# Add edges with connection data
for _, row in connections_df.iterrows():
    if G.has_node(row["From Station"]) and G.has_node(row["To Station"]):
        G.add_edge(
            row["From Station"], row["To Station"],
            connection_type=row["Connection Type"],
            time=row["Time (minutes)"]
        )

# --- Enhanced Visualization ---
plt.figure(figsize=(20, 16))

# Get geographic positions
pos = {node: (G.nodes[node]["lon"], G.nodes[node]["lat"]) for node in G.nodes}

# Create node color/size mapping
node_colors = []
node_sizes = []
for node in G.nodes():
    node_colors.append("red" if G.nodes[node]["status"] == "closed" else "limegreen")
    node_sizes.append(G.nodes[node]["ridership"]/100)  # Scale ridership for visibility

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.8,
    edgecolors="black"
)

# Draw edges with labels
edge_colors = []
edge_styles = []
for u, v, data in G.edges(data=True):
    plt.text((pos[u][0] + pos[v][0])/2,
             (pos[u][1] + pos[v][1])/2,
             f"{data['time']}min\n({data['connection_type']})",
             fontsize=8, ha='center', va='center',
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    # Style edges
    if data["connection_type"] == "Walk":
        edge_colors.append("orange")
        edge_styles.append("dashed")
    else:
        edge_colors.append("navy")
        edge_styles.append("solid")

# Draw styled edges
for (u, v, data), color, style in zip(G.edges(data=True), edge_colors, edge_styles):
    nx.draw_networkx_edges(
        G, pos,
        edgelist=[(u, v)],
        edge_color=color,
        style=style,
        width=1.5,
        alpha=0.6
    )

# Add labels with ridership
label_pos = {k: (v[0]+0.0005, v[1]-0.0005) for k,v in pos.items()}
nx.draw_networkx_labels(
    G, label_pos,
    font_size=8,
    font_weight="bold",
    labels={node: f"{node}\n{G.nodes[node]['ridership']} riders"
           for node in G.nodes}
)

# Add map context
try:
    import contextily as ctx
    ctx.add_basemap(plt.gca(), crs="EPSG:4326", source=ctx.providers.CartoDB.Positron)
except ImportError:
    print("Install contextily for basemap: pip install contextily")

# ADDED: Draw quarter-mile radius circles around closed stops
for node in G.nodes:
    if G.nodes[node]['status'] == 'closed':
        lat = G.nodes[node]['lat']
        lon = G.nodes[node]['lon']

        # Calculate radius in degrees (accounting for latitude)
        miles_per_deg_lat = 69  # Approx miles per degree latitude
        miles_per_deg_lon = 69 * math.cos(math.radians(lat))  # Adjusted for latitude

        # .07 radius in degrees
        radius_deg_lat = 0.07 / miles_per_deg_lat
        radius_deg_lon = 0.07 / miles_per_deg_lon

        # Create and add ellipse (circle in geographic space)
        ellipse = Ellipse(
            (lon, lat),
            width=2 * radius_deg_lon,
            height=2 * radius_deg_lat,
            edgecolor='red',
            facecolor='none',
            linestyle='--',
            alpha=0.7,
            zorder=5
        )
        plt.gca().add_patch(ellipse)

# Add legend
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label='Open Stop',
              markerfacecolor='limegreen', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Closed Stop',
              markerfacecolor='red', markersize=10),
    plt.Line2D([0], [0], color='navy', lw=2, label='Subway Connection'),
    plt.Line2D([0], [0], color='orange', lw=2, linestyle='--', label='Walking Path'),
    # ADDED: Legend entry for quarter-mile radius
    plt.Line2D([0], [0], color='red', linestyle='--', lw=1, label='Quarter Mile Radius')
]

plt.legend(handles=legend_elements, loc='upper right', fontsize=10)
plt.title("Subway Network Visualization with Ridership and Connection Times", fontsize=14)
plt.xlabel("Longitude", fontsize=12)
plt.ylabel("Latitude", fontsize=12)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 3. Dijkstra's Algorithm Method for Ridership Modeling

#### 3.1 Constraint Review With Closed 14-FM and 23-FM Station

In [ ]:
!pip install pulp

In [ ]:
import pandas as pd
import heapq
import os
from collections import defaultdict

# Load data files
file_path = "path/to/project_folder"
stops_file = os.path.join(file_path, "Mapped Stops Within 1 Mile Radius.xlsx")
connections_file = os.path.join(file_path, "Subway and Walking Distance Between Specific Stations.xlsx")

df_stops = pd.read_excel(stops_file, sheet_name='stops_within_1_mile_radius (2)')
df_connections = pd.read_excel(connections_file, sheet_name='Sheet1')

# Build subway graph with transfer penalties
subway_graph = defaultdict(dict)
for _, row in df_connections[df_connections['Connection Type'] == 'Subway'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    from_line = str(row['From Line'])
    to_line = str(row['To Line'])
    base_time = row['Time (minutes)']

    # Apply transfer penalty if lines differ
    if from_line != to_line and from_line != 'nan' and to_line != 'nan':
        time_val = base_time + 2  # +2 min transfer penalty
    else:
        time_val = base_time

    subway_graph[from_station][to_station] = time_val
    subway_graph[to_station][from_station] = time_val

# Build walking graph with inconvenience factor
walk_graph = defaultdict(dict)
for _, row in df_connections[df_connections['Connection Type'] == 'Walk'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    base_time = row['Time (minutes)']
    time_val = base_time * 1.5  # Apply inconvenience factor

    walk_graph[from_station][to_station] = time_val
    walk_graph[to_station][from_station] = time_val

# Dijkstra's algorithm with path tracking
def dijkstra(graph, start, end):
    if start not in graph or end not in graph:
        return float('inf'), []

    distances = {node: float('inf') for node in graph}
    predecessors = {node: None for node in graph}
    distances[start] = 0
    priority_queue = [(0, start)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        if current_distance > distances[current_node]:
            continue
        if current_node == end:
            break
        for neighbor, weight in graph.get(current_node, {}).items():
            distance = current_distance + weight
            if distance < distances.get(neighbor, float('inf')):
                distances[neighbor] = distance
                predecessors[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))

    # Reconstruct path
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = predecessors.get(current, None)
    path.reverse()
    return distances.get(end, float('inf')), path

# Find best path with walk at start or end
def find_best_walk_path(start, destination, max_walk, subway_g, walk_g):
    candidates = []
    path_details = []

    # Walk at start option
    if start in walk_g:
        for neighbor, walk_time in walk_g[start].items():
            if walk_time <= max_walk:
                if neighbor in subway_g and destination in subway_g:
                    subway_time, subway_path = dijkstra(subway_g, neighbor, destination)
                    if subway_time < float('inf'):
                        total_time = walk_time + subway_time
                        full_path = [start] + subway_path
                        candidates.append(total_time)
                        path_details.append({
                            'type': 'walk at start',
                            'time': total_time,
                            'path': full_path,
                            'walk_portion': (start, neighbor, walk_time)
                        })

    # Walk at end option
    if destination in walk_g:
        for neighbor, walk_time in walk_g[destination].items():
            if walk_time <= max_walk:
                if neighbor in subway_g and start in subway_g:
                    subway_time, subway_path = dijkstra(subway_g, start, neighbor)
                    if subway_time < float('inf'):
                        total_time = subway_time + walk_time
                        full_path = subway_path + [destination]
                        candidates.append(total_time)
                        path_details.append({
                            'type': 'walk at end',
                            'time': total_time,
                            'path': full_path,
                            'walk_portion': (neighbor, destination, walk_time)
                        })

    if not candidates:
        # Fallback to subway only
        subway_time, subway_path = dijkstra(subway_g, start, destination)
        return subway_time, subway_path, 'subway only'

    # Find best candidate
    min_time = min(candidates)
    best_path = next(p for p in path_details if p['time'] == min_time)
    return best_path['time'], best_path['path'], best_path['type']

# Northbound analysis from W4-FM
nb_start = "W4-FM"
nb_dests = ["42-123 Times", "42-6 GCT", "34-FM Herald"]
nb_max_walk = 12  # minutes after inconvenience

# Subway-only passengers (60% of northbound)
nb_subway_times = {}
nb_subway_paths = {}
for dest in nb_dests:
    time, path = dijkstra(subway_graph, nb_start, dest)
    nb_subway_times[dest] = time
    nb_subway_paths[dest] = path

# Walking/subway mix passengers (40% of northbound)
nb_walk_mix_times = {}
nb_walk_mix_paths = {}
nb_walk_mix_types = {}
for dest in nb_dests:
    time, path, path_type = find_best_walk_path(nb_start, dest, nb_max_walk, subway_graph, walk_graph)
    nb_walk_mix_times[dest] = time
    nb_walk_mix_paths[dest] = path
    nb_walk_mix_types[dest] = path_type

# Calculate northbound averages
nb_subway_avg = (0.35 * nb_subway_times["42-123 Times"] +
                 0.40 * nb_subway_times["42-6 GCT"] +
                 0.25 * nb_subway_times["34-FM Herald"])

nb_walk_mix_avg = (0.35 * nb_walk_mix_times["42-123 Times"] +
                   0.40 * nb_walk_mix_times["42-6 GCT"] +
                   0.25 * nb_walk_mix_times["34-FM Herald"])

nb_avg = 0.6 * nb_subway_avg + 0.4 * nb_walk_mix_avg

# Southbound analysis
sb_dests = ["14-6 Union", "Bleeker - 6", "14-123", "W4-FM"]
sb_max_walk = 8  # minutes after inconvenience

# Southbound subway-only passengers (70% of southbound)
sb_subway_start = "34-FM Herald"
sb_subway_times = {}
sb_subway_paths = {}
for dest in sb_dests:
    time, path = dijkstra(subway_graph, sb_subway_start, dest)
    sb_subway_times[dest] = time
    sb_subway_paths[dest] = path

# Southbound walking/subway mix passengers (30% of southbound)
sb_walk_mix_times = {}
sb_walk_mix_paths = {}
sb_walk_mix_types = {}
sb_walk_mix_time_details = {}  # Store both times for each destination

# 5% of southbound passengers exit at 42-FM Bryant
sb_start_42 = "42-FM Bryant"
sb_walk_mix_42_times = {}
sb_walk_mix_42_paths = {}
sb_walk_mix_42_types = {}
for dest in sb_dests:
    time, path, path_type = find_best_walk_path(sb_start_42, dest, sb_max_walk, subway_graph, walk_graph)
    sb_walk_mix_42_times[dest] = time
    sb_walk_mix_42_paths[dest] = path
    sb_walk_mix_42_types[dest] = path_type

# 25% of southbound passengers exit at 34-FM Herald
sb_start_34 = "34-FM Herald"
sb_walk_mix_34_times = {}
sb_walk_mix_34_paths = {}
sb_walk_mix_34_types = {}
for dest in sb_dests:
    time, path, path_type = find_best_walk_path(sb_start_34, dest, sb_max_walk, subway_graph, walk_graph)
    sb_walk_mix_34_times[dest] = time
    sb_walk_mix_34_paths[dest] = path
    sb_walk_mix_34_types[dest] = path_type

# Combine southbound walking mix groups (5% + 25% = 30% total)
for dest in sb_dests:
    time_42 = sb_walk_mix_42_times[dest]
    time_34 = sb_walk_mix_34_times[dest]
    # Weighted average: 5/30 from 42-FM Bryant, 25/30 from 34-FM Herald
    weighted_time = (5/30)*time_42 + (25/30)*time_34
    sb_walk_mix_times[dest] = weighted_time

    # Store both paths and times for detailed reporting
    sb_walk_mix_paths[dest] = {
        '42_exit_path': sb_walk_mix_42_paths[dest],
        '34_exit_path': sb_walk_mix_34_paths[dest]
    }
    sb_walk_mix_types[dest] = {
        '42_exit_type': sb_walk_mix_42_types[dest],
        '34_exit_type': sb_walk_mix_34_types[dest]
    }
    sb_walk_mix_time_details[dest] = {
        '42_exit_time': time_42,
        '34_exit_time': time_34,
        'weighted_time': weighted_time
    }

# Calculate southbound averages
sb_subway_avg = (0.30 * sb_subway_times["14-6 Union"] +
                 0.25 * sb_subway_times["Bleeker - 6"] +
                 0.25 * sb_subway_times["14-123"] +
                 0.20 * sb_subway_times["W4-FM"])

sb_walk_mix_avg = (0.30 * sb_walk_mix_times["14-6 Union"] +
                   0.25 * sb_walk_mix_times["Bleeker - 6"] +
                   0.25 * sb_walk_mix_times["14-123"] +
                   0.20 * sb_walk_mix_times["W4-FM"])

sb_avg = (0.7 * sb_subway_avg) + (0.3 * sb_walk_mix_avg)

# Output results
print("="*80)
print("NORTHBOUND PASSENGERS (FROM W4-FM)")
print("="*80)
print("\nSubway-Only Passengers (60% of northbound):")
for dest in nb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {nb_subway_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(nb_subway_paths[dest])}")
print(f"\n  Weighted Average Travel Time: {nb_subway_avg:.2f} minutes")

print("\nWalking/Subway Mix Passengers (40% of northbound):")
for dest in nb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {nb_walk_mix_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(nb_walk_mix_paths[dest])} ({nb_walk_mix_types[dest]})")
print(f"\n  Weighted Average Travel Time: {nb_walk_mix_avg:.2f} minutes")

print("\n" + "-"*40)
print(f"OVERALL NORTHBOUND AVERAGE TRAVEL TIME: {nb_avg:.2f} minutes")
print("-"*40)

print("\n\n" + "="*80)
print("SOUTHBOUND PASSENGERS")
print("="*80)
print("\nSubway-Only Passengers (70% of southbound) - Exited at 34-FM Herald:")
for dest in sb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {sb_subway_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(sb_subway_paths[dest])}")
print(f"\n  Weighted Average Travel Time: {sb_subway_avg:.2f} minutes")

print("\nWalking/Subway Mix Passengers (30% of southbound):")
for dest in sb_dests:
    details = sb_walk_mix_time_details[dest]
    print(f"  To {dest}:")
    print(f"    Average Time: {details['weighted_time']:.2f} minutes")
    print(f"    5% Path (exit at 42-FM Bryant): Time = {details['42_exit_time']:.2f} min | Path: {' → '.join(sb_walk_mix_paths[dest]['42_exit_path'])} ({sb_walk_mix_types[dest]['42_exit_type']})")
    print(f"    25% Path (exit at 34-FM Herald): Time = {details['34_exit_time']:.2f} min | Path: {' → '.join(sb_walk_mix_paths[dest]['34_exit_path'])} ({sb_walk_mix_types[dest]['34_exit_type']})")
print(f"\n  Weighted Average Travel Time: {sb_walk_mix_avg:.2f} minutes")

print("\n" + "-"*40)
print(f"OVERALL SOUTHBOUND AVERAGE TRAVEL TIME: {sb_avg:.2f} minutes")
print("-"*40)

print("\n\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Northbound Average Travel Time: {nb_avg:.2f} minutes")
print(f"Southbound Average Travel Time: {sb_avg:.2f} minutes")

#### 3.2 Constraint Review With Open 14-FM and 23-FM Stations

In [ ]:
import pandas as pd
import heapq
import os
from collections import defaultdict

# Load data files
file_path = "path/to/project_folder"
stops_file = os.path.join(file_path, "Mapped Stops Within 1 Mile Radius.xlsx")
connections_file = os.path.join(file_path, "Subway and Walking Distance Between Specific Stations_Adj for Open Stops.xlsx")

df_stops = pd.read_excel(stops_file, sheet_name='stops_within_1_mile_radius (2)')
df_connections = pd.read_excel(connections_file, sheet_name='Sheet1')

# Build subway graph with transfer penalties
subway_graph = defaultdict(dict)
for _, row in df_connections[df_connections['Connection Type'] == 'Subway'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    transfer = row['Transfer (Y/N)']
    base_time = row['Time (minutes)']

    # Apply transfer penalty only if transfer is marked as 'Y'
    if transfer == 'Y':
        time_val = base_time + 2  # +2 min transfer penalty
    else:
        time_val = base_time

    subway_graph[from_station][to_station] = time_val
    subway_graph[to_station][from_station] = time_val

# Build walking graph without inconvenience factor
walk_graph = defaultdict(dict)
for _, row in df_connections[df_connections['Connection Type'] == 'Walk'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    time_val = row['Time (minutes)']  # No inconvenience factor

    walk_graph[from_station][to_station] = time_val
    walk_graph[to_station][from_station] = time_val

# Dijkstra's algorithm with path tracking
def dijkstra(graph, start, end):
    if start not in graph or end not in graph:
        return float('inf'), []

    distances = {node: float('inf') for node in graph}
    predecessors = {node: None for node in graph}
    distances[start] = 0
    priority_queue = [(0, start)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        if current_distance > distances[current_node]:
            continue
        if current_node == end:
            break
        for neighbor, weight in graph.get(current_node, {}).items():
            distance = current_distance + weight
            if distance < distances.get(neighbor, float('inf')):
                distances[neighbor] = distance
                predecessors[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))

    # Reconstruct path
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = predecessors.get(current, None)
    path.reverse()
    return distances.get(end, float('inf')), path

# Find best path with walk at start or end
def find_best_walk_path(start, destination, max_walk, subway_g, walk_g):
    candidates = []
    path_details = []

    # Walk at start option
    if start in walk_g:
        for neighbor, walk_time in walk_g[start].items():
            if walk_time <= max_walk:
                if neighbor in subway_g:
                    subway_time, subway_path = dijkstra(subway_g, neighbor, destination)
                    if subway_time < float('inf'):
                        total_time = walk_time + subway_time
                        full_path = [start] + subway_path
                        candidates.append(total_time)
                        path_details.append({
                            'type': 'walk at start',
                            'time': total_time,
                            'path': full_path,
                            'walk_portion': (start, neighbor, walk_time)
                        })

    # Walk at end option
    if destination in walk_g:
        for neighbor, walk_time in walk_g[destination].items():
            if walk_time <= max_walk:
                if neighbor in subway_g:
                    subway_time, subway_path = dijkstra(subway_g, start, neighbor)
                    if subway_time < float('inf'):
                        total_time = subway_time + walk_time
                        full_path = subway_path + [destination]
                        candidates.append(total_time)
                        path_details.append({
                            'type': 'walk at end',
                            'time': total_time,
                            'path': full_path,
                            'walk_portion': (neighbor, destination, walk_time)
                        })

    if not candidates:
        # Fallback to subway only
        subway_time, subway_path = dijkstra(subway_g, start, destination)
        return subway_time, subway_path, 'subway only'

    # Find best candidate
    min_time = min(candidates)
    best_path = next(p for p in path_details if p['time'] == min_time)
    return best_path['time'], best_path['path'], best_path['type']

# Northbound analysis from W4-FM
nb_start = "W4-FM"
nb_dests = ["42-123 Times", "42-6 GCT", "34-FM Herald"]
nb_max_walk = 12  # minutes

# Subway-only passengers (60% of northbound)
nb_subway_times = {}
nb_subway_paths = {}
for dest in nb_dests:
    time, path = dijkstra(subway_graph, nb_start, dest)
    nb_subway_times[dest] = time
    nb_subway_paths[dest] = path

# Walking/subway mix passengers (40% of northbound)
nb_walk_mix_times = {}
nb_walk_mix_paths = {}
nb_walk_mix_types = {}
for dest in nb_dests:
    time, path, path_type = find_best_walk_path(nb_start, dest, nb_max_walk, subway_graph, walk_graph)
    nb_walk_mix_times[dest] = time
    nb_walk_mix_paths[dest] = path
    nb_walk_mix_types[dest] = path_type

# Calculate northbound averages
nb_subway_avg = (0.35 * nb_subway_times["42-123 Times"] +
                 0.40 * nb_subway_times["42-6 GCT"] +
                 0.25 * nb_subway_times["34-FM Herald"])

nb_walk_mix_avg = (0.35 * nb_walk_mix_times["42-123 Times"] +
                   0.40 * nb_walk_mix_times["42-6 GCT"] +
                   0.25 * nb_walk_mix_times["34-FM Herald"])

nb_avg = 0.6 * nb_subway_avg + 0.4 * nb_walk_mix_avg

# Southbound analysis from 34-FM Herald
sb_start = "34-FM Herald"
sb_dests = ["14-6 Union", "Bleeker - 6", "14-123", "W4-FM"]
sb_max_walk = 8  # minutes

# Subway-only passengers (70% of southbound)
sb_subway_times = {}
sb_subway_paths = {}
for dest in sb_dests:
    time, path = dijkstra(subway_graph, sb_start, dest)
    sb_subway_times[dest] = time
    sb_subway_paths[dest] = path

# Walking/subway mix passengers (30% of southbound)
sb_walk_mix_times = {}
sb_walk_mix_paths = {}
sb_walk_mix_types = {}
for dest in sb_dests:
    time, path, path_type = find_best_walk_path(sb_start, dest, sb_max_walk, subway_graph, walk_graph)
    sb_walk_mix_times[dest] = time
    sb_walk_mix_paths[dest] = path
    sb_walk_mix_types[dest] = path_type

# Calculate southbound averages
sb_subway_avg = (0.30 * sb_subway_times["14-6 Union"] +
                 0.25 * sb_subway_times["Bleeker - 6"] +
                 0.25 * sb_subway_times["14-123"] +
                 0.20 * sb_subway_times["W4-FM"])

sb_walk_mix_avg = (0.30 * sb_walk_mix_times["14-6 Union"] +
                   0.25 * sb_walk_mix_times["Bleeker - 6"] +
                   0.25 * sb_walk_mix_times["14-123"] +
                   0.20 * sb_walk_mix_times["W4-FM"])

sb_avg = (0.7 * sb_subway_avg) + (0.3 * sb_walk_mix_avg)

# Output results
print("="*80)
print("NORTHBOUND PASSENGERS (FROM W4-FM)")
print("="*80)
print("\nSubway-Only Passengers (60% of northbound):")
for dest in nb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {nb_subway_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(nb_subway_paths[dest])}")
print(f"\n  Weighted Average Travel Time: {nb_subway_avg:.2f} minutes")

print("\nWalking/Subway Mix Passengers (40% of northbound):")
for dest in nb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {nb_walk_mix_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(nb_walk_mix_paths[dest])} ({nb_walk_mix_types[dest]})")
print(f"\n  Weighted Average Travel Time: {nb_walk_mix_avg:.2f} minutes")

print("\n" + "-"*40)
print(f"OVERALL NORTHBOUND AVERAGE TRAVEL TIME: {nb_avg:.2f} minutes")
print("-"*40)

print("\n\n" + "="*80)
print("SOUTHBOUND PASSENGERS (FROM 34-FM HERALD)")
print("="*80)
print("\nSubway-Only Passengers (70% of southbound):")
for dest in sb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {sb_subway_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(sb_subway_paths[dest])}")
print(f"\n  Weighted Average Travel Time: {sb_subway_avg:.2f} minutes")

print("\nWalking/Subway Mix Passengers (30% of southbound):")
for dest in sb_dests:
    print(f"  To {dest}:")
    print(f"    Time: {sb_walk_mix_times[dest]:.2f} minutes")
    print(f"    Path: {' → '.join(sb_walk_mix_paths[dest])} ({sb_walk_mix_types[dest]})")
print(f"\n  Weighted Average Travel Time: {sb_walk_mix_avg:.2f} minutes")

print("\n" + "-"*40)
print(f"OVERALL SOUTHBOUND AVERAGE TRAVEL TIME: {sb_avg:.2f} minutes")
print("-"*40)

print("\n\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Northbound Average Travel Time: {nb_avg:.2f} minutes")
print(f"Southbound Average Travel Time: {sb_avg:.2f} minutes")





### 4. Monte Carlo Algorithm Simulation for Ridership Modeling


#### 4.1 Minimization Simulation with Station 14-FM and 23-FM Closed

In [ ]:
import pandas as pd
import scipy
import random
import math
import heapq
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np

# File paths
file_path = "path/to/project_folder"
stops_file = os.path.join(file_path, "Mapped Stops Within 1 Mile Radius.xlsx")
connections_file = os.path.join(file_path, "Subway and Walking Distance Between Specific Stations.xlsx")

# Load data
df_stops = pd.read_excel(stops_file)
df_connections = pd.read_excel(connections_file)

def haversine(coord1, coord2):
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    R = 3958.8  # Earth radius in miles

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2) * math.sin(dlat/2) +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon/2) * math.sin(dlon/2))
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def generate_points(center, n, radius=0.07):  # radius in miles
    points = []
    for _ in range(n):
        # Convert radius from miles to degrees latitude
        r = radius / 69
        angle = random.uniform(0, 2 * math.pi)
        rand_radius = math.sqrt(random.random())

        dx = r * rand_radius * math.cos(angle)
        dy = r * rand_radius * math.sin(angle) / math.cos(math.radians(center[0]))

        points.append((center[0] + dy, center[1] + dx))
    return points

def get_nearest_stations(commuter_loc, station_coords, n=3):
    distances = []
    for station, coord in station_coords.items():
        dist = haversine(commuter_loc, coord)
        distances.append((station, dist))
    distances.sort(key=lambda x: x[1])
    return [station for station, _ in distances[:n]]

def dijkstra(graph, start, end):
    if start not in graph or end not in graph:
        return float('inf'), []

    distances = {node: float('inf') for node in graph}
    predecessors = {node: None for node in graph}
    distances[start] = 0
    priority_queue = [(0, start)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        if current_distance > distances[current_node]:
            continue
        if current_node == end:
            break
        for neighbor, weight in graph.get(current_node, {}).items():
            distance = current_distance + weight
            if distance < distances.get(neighbor, float('inf')):
                distances[neighbor] = distance
                predecessors[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))

    # Reconstruct path
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = predecessors.get(current, None)
    path.reverse()
    return distances.get(end, float('inf')), path

# Subway Graph
subway_graph = defaultdict(dict)
for _, row in df_connections[df_connections['Connection Type'] == 'Subway'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    from_line = str(row['From Line'])
    to_line = str(row['To Line'])
    base_time = row['Time (minutes)']

    # Apply transfer penalty if lines differ
    if from_line != to_line and from_line != 'nan' and to_line != 'nan':
        time_val = base_time + 2  # +2 min transfer penalty
    else:
        time_val = base_time

    subway_graph[from_station][to_station] = time_val
    subway_graph[to_station][from_station] = time_val

# Build subway graph without closed stations
subway_graph = defaultdict(dict)
closed_stations = {"14-FM", "23-FM"}

for _, row in df_connections[df_connections['Connection Type'] == 'Subway'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    transfer = row['Transfer (Y/N)']
    base_time = row['Time (minutes)']

    # Skip connections involving closed stations
    if from_station in closed_stations or to_station in closed_stations:
        continue

    # Apply transfer penalty if needed
    if transfer == 'Y':
        time_val = base_time + 2  # +2 min transfer penalty
    else:
        time_val = base_time

    # Add to graph (bidirectional)
    subway_graph[from_station][to_station] = time_val
    subway_graph[to_station][from_station] = time_val


EXCEPTIONS = {
    "Bleeker - 6"
}

def normalize_station_name(name):
    if name in EXCEPTIONS:
        return name
    return re.sub(r'\s*-\s*', '-', name)

# Create station data lookup
station_coords = {}
for _, row in df_stops.iterrows():
    name = row['Name on Map']
    station_coords[name] = (row['stop_lat'], row['stop_lon'])


closed_station_info = {
    "14-FM": {
        "coords": (40.738228, -73.996209),
    },
    "23-FM": {
        "coords": (40.742878, -73.992821),
    }
}

closed_station_coords = [info["coords"] for info in closed_station_info.values()]
# Map out stations and average capacity (Starting State)
stations_capacity = {"42 - 123 Times": 41813,"42 - FM Bryant": 20083,"42 - 6 GCT": 44665,"34 - 123 Penn": 13405, "34 - FM Herald": 30407,
            "28 - RW": 3616, "23 - 123": 3589, "23 - RW": 5887, "14 - 123": 11737, "14 - 6 Union": 27090,
            "W4 - FM": 10907, "Bleeker - 6": 12273, "Closed Stop: 14-FM":100, "Closed Stop: 23-FM":100}


# Define our capacity limit
stations_capacity_limit = stations_capacity.copy()
stations_capacity_limit = {
    key: val * 1.1 for key, val in stations_capacity.items()
}

# Station leaving rate. Based on capacity size. Higher the size, higher the rate
station_leaving_rate = {"42 - 123": .07,"42 - FM Bryant": .06,"42 - 6 GCT": .07, "34 - 123 Penn": .04, "34 - FM Herald": .06,
            "28 - RW": .02, "23 - 123": .02, "23 - RW": .03, "14 - 123": .04, "14 - 6 Union": .05,
            "W4 - FM": .03, "Bleeker - 6": .06}


# Create weights to assign destinations for commuters
total_station_capacity = sum(stations_capacity.values())
station_weights = {station: cap / total_station_capacity for station, cap in stations_capacity.items()}
stations = list(station_weights.keys())
weights = list(station_weights.values())


# Define our commuter departure distribution (Beta Distribution)
alpha = 2.625
beta = 3.375
# Evening Rush hour 4-8 PM
total_minutes = 240
# Assumption based on the two stations that closed
total_commuters = 25000
# Every wave will be 30 minutes apart
interval = 30
time_steps = list(range(0, total_minutes + 1, interval))
# Define a set for closed stations (nodes)
closed = set()
closed.add("Closed Stop: 14-FM")
closed.add("Closed Stop: 23-FM")

def pick_open_station(stations, weights, closed):
    while True:
        station = random.choices(stations, weights=weights, k=1)[0]
        if station not in closed:
            return station

results = []
# For each wave
for wave_i in range(1, len(time_steps)):
    t0 = time_steps[wave_i-1] / total_minutes
    t1 = time_steps[wave_i] / total_minutes

    wave_commuters = round(total_commuters * (scipy.stats.beta.cdf(t1, alpha, beta) - scipy.stats.beta.cdf(t0, alpha, beta)))

    # print(f"Wave {wave_i} | Time: {time_steps[wave_i-1]}–{time_steps[wave_i]} mins | Commuters: {wave_commuters}")

    time = []
    # Simulate each commuter per wave
    for commuter_i in range(wave_commuters):
      destination = pick_open_station(stations,weights,closed)
      center = random.choice(closed_station_coords)
      passenger_point = generate_points(center, 1, radius=0.07)[0]
      nearest_stations = get_nearest_stations(passenger_point, station_coords, n=3)

      for station in nearest_stations:
        if station not in closed:
          if stations_capacity[station] < stations_capacity_limit[station]:
              start = station
              stations_capacity[station] += 1

            # Run Dijkstra from start to destination
              if start == destination:
                destination = random.choices(stations, weights = weights, k=1)[0]
              else:
                distance, path = dijkstra(subway_graph, normalize_station_name(start), normalize_station_name(destination))
                time.append(distance)
                # print(f"Commuter {commuter_i}: {start} ➝ {destination}, Path: {path}, Distance: {distance}")
                break
          else:
              # print(f"{station} is overcrowded.")
              closed.add(station)
    if time:
      average_time = np.mean(time)
      results.append({
          "Wave": wave_i,
          "StartTime": time_steps[wave_i-1],
          "EndTime": time_steps[wave_i],
          "AverageTime": average_time,
          "Commuters": wave_commuters
      })

closed_station_df_results = pd.DataFrame(results)
print(closed_station_df_results)




# Adjust the capacity rates after each wave
for station in stations_capacity_limit:
    leaving_rate = station_leaving_rate.get(station, 0.04)
    departures = stations_capacity_limit[station] * leaving_rate
    stations_capacity[station] = max(0, stations_capacity_limit[station] - departures)

#### 4.2 Minimization Simulation with Station 14-FM and 23-FM Open

In [ ]:
# Control (No Closure of stations to compare average commute times)
import pandas as pd
import scipy
import random
import math
import heapq
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np

# File paths
file_path = "path/to/project_folder"
stops_file = os.path.join(file_path, "Mapped Stops Within 1 Mile Radius.xlsx")
connections_file = os.path.join(file_path, "Subway and Walking Distance Between Specific Stations_Adj for Open Stops.xlsx")

# Load data
df_stops = pd.read_excel(stops_file)
df_connections = pd.read_excel(connections_file)

def haversine(coord1, coord2):
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    R = 3958.8  # Earth radius in miles

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2) * math.sin(dlat/2) +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon/2) * math.sin(dlon/2))
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def generate_points(center, n, radius=0.07):  # radius in miles
    points = []
    for _ in range(n):
        # Convert radius from miles to degrees latitude
        r = radius / 69
        angle = random.uniform(0, 2 * math.pi)
        rand_radius = math.sqrt(random.random())

        dx = r * rand_radius * math.cos(angle)
        dy = r * rand_radius * math.sin(angle) / math.cos(math.radians(center[0]))

        points.append((center[0] + dy, center[1] + dx))
    return points

def get_nearest_stations(commuter_loc, station_coords, n=3):
    distances = []
    for station, coord in station_coords.items():
        dist = haversine(commuter_loc, coord)
        distances.append((station, dist))
    distances.sort(key=lambda x: x[1])
    return [station for station, _ in distances[:n]]

def dijkstra(graph, start, end):
    if start not in graph or end not in graph:
        return float('inf'), []

    distances = {node: float('inf') for node in graph}
    predecessors = {node: None for node in graph}
    distances[start] = 0
    priority_queue = [(0, start)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        if current_distance > distances[current_node]:
            continue
        if current_node == end:
            break
        for neighbor, weight in graph.get(current_node, {}).items():
            distance = current_distance + weight
            if distance < distances.get(neighbor, float('inf')):
                distances[neighbor] = distance
                predecessors[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))

    # Reconstruct path
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = predecessors.get(current, None)
    path.reverse()
    return distances.get(end, float('inf')), path

# Subway Graph
subway_graph = defaultdict(dict)
for _, row in df_connections[df_connections['Connection Type'] == 'Subway'].iterrows():
    from_station = row['From Station']
    to_station = row['To Station']
    from_line = str(row['From Line'])
    to_line = str(row['To Line'])
    base_time = row['Time (minutes)']

    # Apply transfer penalty if lines differ
    if from_line != to_line and from_line != 'nan' and to_line != 'nan':
        time_val = base_time + 2  # +2 min transfer penalty
    else:
        time_val = base_time

    subway_graph[from_station][to_station] = time_val
    subway_graph[to_station][from_station] = time_val


EXCEPTIONS = {
    "Bleeker - 6",
    "Closed Stop: 14-FM",
    "Closed Stop: 23-FM"
}

def normalize_station_name(name):
    if name in EXCEPTIONS:
        return name
    return re.sub(r'\s*-\s*', '-', name)

# Change the stations names
name_changes = {
    'Closed Stop: 14-FM' : '14-FM',
     'Closed Stop: 23-FM' : '23-FM'
}

df_stops['Name on Map'] = df_stops['Name on Map'].replace(name_changes)


# Create station data lookup
station_coords = {}
for _, row in df_stops.iterrows():
    name = row['Name on Map']
    station_coords[name] = (row['stop_lat'], row['stop_lon'])



closed_station_coords = [info["coords"] for info in closed_station_info.values()]
# Map out stations and average capacity (Starting State)
stations_capacity = {"42 - 123 Times": 41813,"42 - FM Bryant": 20083,"42 - 6 GCT": 44665,"34 - 123 Penn": 13405, "34 - FM Herald": 30407,
            "28 - RW": 3616, "23 - 123": 3589, "23 - RW": 5887, "14 - 123": 11737, "14 - 6 Union": 27090,
            "W4 - FM": 10907, "Bleeker - 6": 12273, "14-FM":11737, "23-FM":7869}


# Define our capacity limit
stations_capacity_limit = stations_capacity.copy()
stations_capacity_limit = {
    key: val * 1.1 for key, val in stations_capacity.items()
}

# Station leaving rate. Based on capacity size. Higher the size, higher the rate
station_leaving_rate = {"42 - 123": .07,"42 - FM Bryant": .06,"42 - 6 GCT": .07, "34 - 123 Penn": .04, "34 - FM Herald": .06,
            "28 - RW": .02, "23 - 123": .02, "23 - RW": .03, "14 - 123": .04, "14 - 6 Union": .05,
            "W4 - FM": .03, "Bleeker - 6": .06, "14-FM": .04, "23-FM": .02 }


# Create weights to assign destinations for commuters
total_station_capacity = sum(stations_capacity.values())
station_weights = {station: cap / total_station_capacity for station, cap in stations_capacity.items()}
stations = list(station_weights.keys())
weights = list(station_weights.values())



# Define our commuter departure distribution (Beta Distribution)
# Different alpha and beta parameters

# alpha + beta = 2, a = .875, beta = 1.125
# alpha + beta = 4, a = 1.75, beta = 2.25
# alpha + beta = 8, a = 3.5, beta = 4.5
# alpha + beta = 10, a = 4.375, beta = 5.625
alpha = 2.625
beta = 3.375
# Evening Rush hour 4-8 PM
total_minutes = 240
# Assumption based on the two stations that closed
total_commuters = 25000
# Every wave will be 30 minutes apart
interval = 30
time_steps = list(range(0, total_minutes + 1, interval))

# Take out the set
def pick_open_station(stations, weights, closed):
    while True:
        station = random.choices(stations, weights=weights, k=1)[0]
        if station not in closed:
            return station

result = []
# For each wave
for wave_i in range(1, len(time_steps)):
    t0 = time_steps[wave_i-1] / total_minutes
    t1 = time_steps[wave_i] / total_minutes

    wave_commuters = round(total_commuters * (scipy.stats.beta.cdf(t1, alpha, beta) - scipy.stats.beta.cdf(t0, alpha, beta)))

    # print(f"Wave {wave_i} | Time: {time_steps[wave_i-1]}–{time_steps[wave_i]} mins | Commuters: {wave_commuters}")

    time = []
    # Simulate each commuter per wave
    for commuter_i in range(wave_commuters):
      destination = pick_open_station(stations,weights,closed)
      center = random.choice(closed_station_coords)
      passenger_point = generate_points(center, 1, radius=0.07)[0]
      nearest_stations = get_nearest_stations(passenger_point, station_coords, n=3)

      for station in nearest_stations:
        if station not in closed:
          if stations_capacity[station] < stations_capacity_limit[station]:
              start = station
              stations_capacity[station] += 1

            # Run Dijkstra from start to destination
              if start == destination:
                destination = random.choices(stations, weights = weights, k=1)[0]
              else:
                distance, path = dijkstra(subway_graph, normalize_station_name(start), normalize_station_name(destination))
                time.append(distance)
                # print(f"Commuter {commuter_i}: {start} ➝ {destination}, Path: {path}, Distance: {distance}")
                break
          else:
              # print(f"{station} is overcrowded.")
              closed.add(station)

    if time:
      average_time = np.mean(time)
      result.append({
          "Wave": wave_i,
          "StartTime": time_steps[wave_i-1],
          "EndTime": time_steps[wave_i],
          "AverageTime": average_time,
          "Commuters": wave_commuters
      })


open_station_df_results = pd.DataFrame(result)
print(open_station_df_results)



# Adjust the capacity rates after each wave
for station in stations_capacity_limit:
    leaving_rate = station_leaving_rate.get(station, 0.04)
    departures = stations_capacity_limit[station] * leaving_rate
    stations_capacity[station] = max(0, stations_capacity_limit[station] - departures)

In [ ]:
merged_df = closed_station_df_results.merge(open_station_df_results, on=['Wave', 'StartTime', 'EndTime', 'Commuters'], suffixes=('_closed', '_open'))
merged_df['Difference'] = merged_df['AverageTime_closed'] - merged_df['AverageTime_open']
print(merged_df)

#### 4.3 Sensitivity Analysis for distribution for each alpha and beta

In [ ]:
# Create distribution visualizations for each alpha and beta
import scipy.stats
import matplotlib.pyplot as plt
import numpy as np

parameters = [(0.875, 1.125), (1.75, 2.25), (2.625, 3.375), (3.5, 4.5), (4.375, 5.625),(5.25,6.75)]
total_commuters = 25000
total_minutes = 240
interval = 30
time_steps = list(range(0, total_minutes + 1, interval))

# Create wave labels
wave_labels = [
    "4:00-4:30", "4:30-5:00", "5:00-5:30", "5:30-6:00",
    "6:00-6:30", "6:30-7:00", "7:00-7:30", "7:30-8:00"
]

rows, cols = 2, 3
fig, axes = plt.subplots(rows, cols, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i, (alpha, beta) in enumerate(parameters):
    wave_commuters = []
    for wave_i in range(1, len(time_steps)):
        t0 = time_steps[wave_i - 1] / total_minutes
        t1 = time_steps[wave_i] / total_minutes
        commuters_in_wave = round(
            total_commuters * (scipy.stats.beta.cdf(t1, alpha, beta) - scipy.stats.beta.cdf(t0, alpha, beta))
        )
        wave_commuters.append(commuters_in_wave)

    ax = axes[i]
    # Bar graph
    ax.bar(time_steps[1:], wave_commuters, width=interval*0.9, alpha=0.6, color='skyblue', edgecolor='black')

    # Outline
    x_smooth = np.linspace(0, total_minutes, 300)
    y_smooth = total_commuters * scipy.stats.beta.pdf(x_smooth / total_minutes, alpha, beta) * (interval / total_minutes)
    ax.plot(x_smooth, y_smooth, color='darkblue', linewidth=2)

    ax.set_title(f"α={alpha}, β={beta}")
    ax.set_xlabel("Waves")
    ax.set_ylabel("Commuters")
    ax.grid(True, linestyle='--', alpha=0.5)

    # Set custom x-axis ticks and labels
    ax.set_xticks(time_steps[1:])  # Use the center of each interval
    ax.set_xticklabels(wave_labels, rotation=45, ha='right')

plt.tight_layout()
plt.suptitle("Commuter Distribution with Different α and β Values", fontsize=16, y=1.02)
plt.show()
